In [4]:
import joblib
from Feature_Engineering import FraudFeatureEngineer
from Feature_selection import ColumnSelector

fe = joblib.load("feature_engineering.pkl")
selector = joblib.load("feature_selection.pkl")
model = joblib.load("xgb_model_with_threshold.pkl")

In [5]:
import pandas as pd

df = pd.read_csv("df_target_null.csv")

# Convert datetime columns
df["date"] = pd.to_datetime(df["date"])
df["acct_open_date"] = pd.to_datetime(df["acct_open_date"])

In [6]:
df_fe = fe.transform(df)

c:\Users\Geeks2_PC10\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Geeks2_PC10\Documents\Project Sim Dataset\Training Model final\Feature_Engineering.py:122: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.set_index('date')['abs_amount']


In [7]:
df_selected = selector.transform(df_fe)

In [9]:
print(type(model))

<class 'dict'>


In [10]:
print(model.keys())

dict_keys(['model', 'threshold'])


In [11]:
xgb_model = model["model"]
threshold = model["threshold"]

In [12]:
proba = xgb_model.predict_proba(df_selected)[:, 1]
pred = (proba >= threshold).astype(int)

In [13]:
df["fraud_probability"] = proba
df["fraud_prediction"] = pred

In [14]:
df

,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,birth_month,gender,address,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,fraud_probability,fraud_prediction
0,7475331,2022-06-12 09:33:41,430,2860,3600.00,Swipe Transaction,27092,Bloemfontein,Free State,9300.0,...,5,Female,"573 Durban Road, East London",471024,960300,2316168,685,5,0.000581,0
1,7475334,2024-05-06 13:03:32,1556,2972,1386.00,Swipe Transaction,59935,Polokwane,Limpopo,700.0,...,7,Female,"405 Pretoria Street, Johannesburg",426222,868986,1982754,740,4,0.000369,0
2,7475336,2024-03-21 19:20:35,335,5131,4708.44,Online Transaction,50292,ONLINE,ONLINE,0.0,...,7,Female,"641 Jan Smuts Avenue, Cape Town",498528,1016406,1198170,688,3,0.000508,0
3,7475337,2022-09-12 15:47:58,351,1112,193.32,Swipe Transaction,3864,Polokwane,Limpopo,700.0,...,9,Female,"493 Cape Road, Cape Town",248580,308700,6750,807,6,0.000302,0
4,7475343,2023-09-15 00:30:52,1634,2464,19.62,Swipe Transaction,20519,Kimberley,Northern Cape,8300.0,...,3,Male,"69 Swart Street, Pietermaritzburg",181638,370386,1082736,825,4,0.001094,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4243216,23209451,2023-04-09 09:31:22,1535,187,623.34,Chip Transaction,50524,Durban,KwaZulu-Natal,4000.0,...,8,Male,"860 Church Street, Pretoria",294048,599598,1184796,590,2,0.000053,0
4243217,23209454,2022-05-18 11:11:19,1603,1313,288.72,Chip Transaction,44211,Johannesburg,Gauteng,2000.0,...,12,Male,"472 Durban Road, Cape Town",342648,698616,293958,762,4,0.002621,0
4243218,23209457,2023-06-12 07:02:05,461,5482,973.80,Chip Transaction,75936,East London,Eastern Cape,5200.0,...,12,Female,"241 Voortrekker Road, Durban",577404,1177254,1441026,784,4,0.000014,0
4243219,23209464,2023-07-23 01:05:29,1508,3279,593.46,Chip Transaction,43293,Cape Town,Western Cape,8000.0,...,4,Female,"495 Durban Road, Cape Town",344250,701982,1167750,747,4,0.000087,0


In [15]:
df_fraud = df[df["fraud_prediction"] == 1]

df_fraud

,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,birth_month,gender,address,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,fraud_probability,fraud_prediction
204,7476035,2022-01-27 11:54:50,1315,5434,17.82,Swipe Transaction,14528,Nelspruit,Mpumalanga,1200.0,...,8,Female,"422 Cape Road, Durban",287568,586368,0,753,4,0.965690,1
485,7477074,2023-06-10 00:38:33,866,2110,239.04,Swipe Transaction,10126,Port Elizabeth,Eastern Cape,6000.0,...,3,Male,"768 Main Road, Cape Town",546498,1114272,2256444,639,3,0.294169,1
777,7478161,2022-12-28 07:00:54,933,2523,427.14,Swipe Transaction,45926,Cape Town,Western Cape,8000.0,...,5,Male,"270 Voortrekker Road, Durban",382212,779220,1624284,690,4,0.262646,1
4045,7490071,2024-11-11 18:59:09,1452,3801,1404.00,Swipe Transaction,50867,Nelspruit,Mpumalanga,1200.0,...,5,Female,"529 Swart Street, Durban",1710702,3487914,4348278,660,1,0.316393,1
4060,7490116,2022-07-17 04:17:00,1507,3449,-1062.00,Swipe Transaction,59935,Durban,KwaZulu-Natal,4000.0,...,12,Female,"397 King George Street, Cape Town",391482,798192,1490796,744,3,0.203552,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4240659,23199639,2023-11-06 01:59:50,1117,4476,2257.56,Chip Transaction,18131,East London,Eastern Cape,5200.0,...,4,Female,"745 Jan Smuts Avenue, Pretoria",244476,498438,440640,687,2,0.435704,1
4241149,23201483,2024-01-06 14:33:18,1772,56,1040.22,Chip Transaction,20519,Bloemfontein,Free State,9300.0,...,10,Male,"36 Pretoria Street, Pretoria",442854,902934,1749384,761,4,0.374956,1
4242009,23204732,2023-03-04 01:05:27,1144,5153,644.04,Chip Transaction,88646,Pietermaritzburg,KwaZulu-Natal,3200.0,...,12,Female,"123 Voortrekker Road, Bloemfontein",269262,548910,1274850,773,4,0.336838,1
4242361,23206030,2024-07-08 09:08:40,1227,3545,75.78,Chip Transaction,59935,Polokwane,Limpopo,700.0,...,11,Female,"39 Pretoria Street, East London",594558,1212192,126882,804,4,0.971566,1
